In [ ]:
!g++ -O3 -Wall -shared -std=c++17 -DMS_WIN64 wrapper.cpp tensor.cpp model.cpp layer.cpp loss.cpp optimizer.cpp opencl_runtime.cpp opencl.dll  -IC:\Users\user\AppData\Local\Programs\Python\Python313\Include "-Ic:\Users\user\Documents\vscode\monke_deep_learning_frame\.venv\Lib\site-packages\pybind11\include"  -I"." -L. -LC:\Users\user\AppData\Local\Programs\Python\Python313\libs -lpython313 -lOpenCL -static-libgcc -static-libstdc++ -o monke_frame.pyd

In [ ]:
import os
import sys

# 1. 清理舊路徑並重新指向
current_dir = os.getcwd()
# 這是你 MSYS2 g++ 存放依賴 DLL 的地方
mingw_bin = r"C:\msys64\ucrt64\bin" 

if os.name == 'nt':
    # A. 加入目前資料夾 (找 OpenCL.dll)
    os.add_dll_directory(current_dir)
    
    # B. 加入 MSYS2 執行庫路徑 (找 libstdc++-6.dll 等)
    if os.path.exists(mingw_bin):
        os.add_dll_directory(mingw_bin)
        print(f"✅ 已加入 MinGW 路徑: {mingw_bin}")
    else:
        print("⚠️ 找不到 MinGW 路徑，請確認你的 msys64 安裝位置")

# 2. 嘗試載入
try:
    import monke_frame
    print("✅ 【成功載入】這一次真的解決了！")
except ImportError as e:
    print(f"❌ 載入失敗：{e}")
    # 檢查是否缺少特定 DLL
    import subprocess
    # 用 ldd 查看依賴項 (如果環境支援)
    print("\n--- 依賴項檢查 ---")
    os.system(f'dumpbin /dependents monke_frame.pyd') # 如果你有 VC 工具

In [ ]:
import monke_frame

monke_frame.test_print()
monke_frame.initialize_opencl()

In [ ]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist

# 自動下載並加載
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# 標準化：把 0~255 轉成 0.0~1.0 (float32 是 OpenCL 的最愛)
train_images = train_images.astype(np.float32) / 255.0
test_images = test_images.astype(np.float32) / 255.0

# 檢查形狀：應該是 (60000, 28, 28)
print(f"訓練資料形狀: {train_images.shape}")

In [ ]:
import matplotlib.pyplot as plt

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


num_samples = 60000

inputs = []
reals = []

for i in range(num_samples):
    # Fashion-MNIST 原始是 (28, 28)，增加通道維度變 (1, 28, 28)
    img_gray = train_images[i].reshape(1, 28, 28).astype(np.float32)

    if i < 5:
        plt.figure(figsize=(2, 2))
        plt.imshow(img_gray.squeeze(), cmap='gray')
        plt.title(f"{class_names[train_labels[i]]}") 
        plt.axis('off')
        plt.show()

    # 轉換為 Tensor 並移至 GPU (你的 Monke Frame)
    input_tensor = monke_frame.Tensor(img_gray)
    inputs.append(input_tensor)

    # 處理標籤 (One-hot encoding)
    label_tensor = monke_frame.Tensor(np.zeros(10, dtype=np.float32))
    label_tensor.set([int(train_labels[i])], 1.0)
    
    reals.append(label_tensor)

In [ ]:
model = monke_frame.Model()

model.add_layer(monke_frame.Convolution(1,28,32,5))# 1x28x28 -> 32x24x24
model.add_layer(monke_frame.ReLU(32*24*24,0.01))

model.add_layer(monke_frame.Convolution(32,24,64,3))# 32x24x24 -> 64x22x22
model.add_layer(monke_frame.ReLU(64*22*22,0.01))

model.add_layer(monke_frame.Convolution(64,22,128,3))# 64x22x22 -> 128x20x20
model.add_layer(monke_frame.ReLU(128*20*20,0.01))

model.add_layer(monke_frame.Convolution(128,20,256,3))# 128x20x20 -> 256x18x18
model.add_layer(monke_frame.ReLU(256*18*18,0.01))


model.add_layer(monke_frame.Dense(82944, 128)) # 82944 -> 128
model.add_layer(monke_frame.ReLU(128,0.01))

model.add_layer(monke_frame.Dense(128, 10)) # 128 -> 10
model.add_layer(monke_frame.Softmax(10))


model.compile(monke_frame.CrossEntropy(10), monke_frame.Adam(0.9,0.99,1e-9))

In [ ]:
batch_size = 200
inputs_batched_ptr = model.auto_batching(inputs, batch_size)
outputs_batched_ptr = model.auto_batching(reals, batch_size)

In [ ]:
print(len(inputs_batched_ptr))
for input_batch in inputs_batched_ptr:
    input_batch.toCPU()
    input_batch.print(28*28)
    input_batch.toGPU()

In [ ]:
print(len(outputs_batched_ptr))
for output_batch in outputs_batched_ptr:
    output_batch.toCPU()
    output_batch.print(10)
    output_batch.toGPU()

In [ ]:
batch_sizes = []
for i in range(len(inputs_batched_ptr)):
    batch_sizes.append(inputs_batched_ptr[i].getShape()[0])

In [ ]:
for b in batch_sizes:
    print(b)

In [ ]:
model.setIntermediateInputs(batch_size)

In [ ]:
import matplotlib.pyplot as plt

lr = 0.0001
pre_loss = 0

lr_history = []
loss_history = []

for epoch in range(100):
    print(f"epoch {epoch}")
    loss = 0.0

    # 紀錄目前的 lr
    lr_history.append(lr)

    for i in range(len(batch_sizes)):
        loss += model.train(inputs_batched_ptr[i], outputs_batched_ptr[i], batch_sizes[i], lr)
    
    loss /= len(batch_sizes)

    print(f"{loss}({loss-pre_loss}) lr={lr}")

    # 紀錄目前的 loss (方便一起觀察)
    loss_history.append(loss)
    if(loss <= 0.1):
        break
    

    if(loss > pre_loss * 1.01 and epoch > 0): # 避免 epoch 0 就砍 lr
        lr /= 2

    pre_loss = loss

    lr *= 1.001

# --- 訓練結束後：輸出圖表 ---
plt.figure(figsize=(12, 5))

# 畫 Learning Rate 圖
plt.subplot(1, 2, 1)
plt.plot(lr_history, color='blue', label='Learning Rate')
plt.title('Learning Rate over Epochs')
plt.xlabel('Epoch')
plt.ylabel('LR Value')
plt.grid(True)
plt.legend()

# 畫 Loss 圖 (建議一起看才有意義)
plt.subplot(1, 2, 2)
plt.plot(loss_history, color='red', label='Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.setIntermediateInputs(1) # Set intermediate inputs for single prediction
count = 0
for i in range(60000):
    pred = model.predict(inputs[i])
    pred.toCPU()
    print(f"Sample {i} Prediction:")
    max = 0.0
    pred_index = 0
    for j in range(10):
        if(pred.get([0,j]) > max):
            max = pred.get([0,j])
            pred_index = j
    real_index = 0.0
    for j in range(10):
        if(reals[i].get([j]) == 1.0):
            real_index = j
    print(f"Predicted Label: {pred_index}, Real Label: {real_index}")
    if(pred_index == real_index):
        count += 1
    else:
        print("❌ 錯誤")
print(f"Total correct predictions: {count}/60000 ({(count/60000)*100:.2f}%)")

In [ ]:
# 載入測試集 (test_images 是 10,000 筆)
#(test_images, test_labels)

# 拿新的 100 筆 (例如從 0 到 100)
num_test = 10000
# 從測試集中隨機抽樣
test_indices = np.random.choice(len(test_images), num_test, replace=False)

test_subset_images = test_images[test_indices]
test_subset_labels = test_labels[test_indices]

test_inputs = []
test_reals = []
for i in range(num_test):
    # 同樣轉成 (1, 28, 28)
    img_gray = test_subset_images[i].reshape(1, 28, 28).astype(np.float32)

    t_input = monke_frame.Tensor(img_gray)
    
    test_inputs.append(t_input)

    # 標籤也存起來備用
    t_real = monke_frame.Tensor(np.zeros(10, dtype=np.float32))
    t_real.set([int(test_subset_labels[i])], 1.0)
    
    test_reals.append(t_real)

print(f"✅ 測試集準備完成：{num_test} 筆新資料")

In [ ]:
import matplotlib.pyplot as plt
import math

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

test_count = 0
wrong_cases = [] # 用來存 (圖片, 預測值, 真實值)
model.setIntermediateInputs(0) # Set intermediate inputs for single prediction

print("--- 正在進行測試並分析錯誤 ---")

for i in range(num_test):
    # 1. 模型預測
    pred = model.predict(test_inputs[i])
    pred.toCPU() # 確保在 CPU 上讀取結果
    
    temp_max = 0.0
    predict_label = 0
    for j in range(10):
        if(pred.get([j]) > temp_max):
            temp_max = pred.get([j])
            predict_label = j
    temp_max = 0.0
    true_label = 0
    for j in range(10):
        if(test_reals[i].get([j]) == 1.0):
            true_label = j

    

    # 3. 判斷對錯
    if predict_label == true_label:
        test_count += 1
    else:
        # 紀錄錯誤的資料：原始圖片、預測值、真實答案
        wrong_cases.append((test_subset_images[i], predict_label, true_label))

# --- 輸出準確率 ---
accuracy = (test_count / num_test) * 100
print(f"\n✅ 測試完成！")
print(f"總測試筆數: {num_test}")
print(f"正確筆數: {test_count}")
print(f"錯誤筆數: {len(wrong_cases)}")
print(f"最終準確率: {accuracy:.2f}%")

# --- 顯示錯誤的圖片 ---
if len(wrong_cases) > 0:
    print(f"\n--- 正在顯示錯誤案例 (共 {len(wrong_cases)} 張) ---")

    # 計算畫布網格，例如 80 張錯就排成 9x9
    num_wrong = len(wrong_cases)
    cols = 10 # 固定每行顯示 10 張
    rows = math.ceil(num_wrong / cols)

    plt.figure(figsize=(15, rows * 1.8)) # 根據錯誤數量自動調整高度

    for idx, (img, p_lab, r_lab) in enumerate(wrong_cases):
        plt.subplot(rows, cols, idx + 1)
        plt.imshow(img, cmap='gray')
        # P: Predict, R: Real
        plt.title(f"P:{class_names[p_lab]} R:{class_names[r_lab]}", color='red', fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("太厲害了！全對，沒有錯誤案例可以顯示。")